# 🚀 QTrader Analyst Platform | v4.0
Welcome to the unified quantitative research terminal. This platform bridges the gap between raw data, alpha research, and live execution.

### 📂 Specialized Workflows
1. **`researcher/`**: Feature Engineering, Regime detection, and ML Experiments.
2. **`analyst/`**: Backtest reporting, Risk metrics, and Portfolio optimization.
3. **`trader/`**: Live execution audit and Bot monitoring.

---

In [1]:
from qtrader.output.analyst import AnalystSession, RoleContext
import plotly.express as px
import polars as pl

session = AnalystSession(role=RoleContext.RESEARCHER)
session.info()

PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

🔬 Quant Researcher Workflow
  1. load_ohlcv → raw OHLCV
  2. add_rolling_features → compute & store features via FeatureStore
  3. run_alpha_score → forward-return scoring
  4. Use MLflow for experiment tracking (see notebooks/researcher/)
  5. load_features → pull pre-computed features

  📓 Notebooks: notebooks/researcher/01_Feature_Lab.ipynb, ...



## ⚡ Quick Discovery
Load data and visualize volatility patterns instantly.

In [12]:
symbol = "BNB-USD"
timeframe = "1m"

try:
    df = session.load_live_ohlcv(symbol, timeframe , days = 1)
except Exception:
    df = session.sample_ohlcv(symbol="DOGE", days=365)

df = session.make_returns(df)
df = session.add_rolling_features(df)

fig = px.line(df.to_pandas(), x="timestamp", y="close", title=f"{symbol} Price Action")
fig.update_layout(**PLOTLY_DARK)
fig.show()

## 📈 Historical Data Extraction (30 Days)
Fetch a large volume of historical data. The `CoinbaseMarketDataClient` automatically handles pagination (300 candles per batch) and anti-block throttling (150ms per request) with exponential backoff.

In [13]:
from qtrader.input.data.market.coinbase_market import CoinbaseMarketDataClient
from datetime import datetime, timedelta
from qtrader.core.config import Config
import plotly.graph_objects as go
import plotly.express as px
import polars as pl

client = CoinbaseMarketDataClient()
symbol = "BNB-USD"
granularity = "ONE_MINUTE"

end_dt = datetime.now(Config.tz)
start_dt = end_dt - timedelta(days=1)  # Reduce to 1 day for a clear candlestick view

print(f"Fetching 24h of 1m data for {symbol} to visualize candlesticks...")
df_hist = client.get_candles(symbol, granularity, start=start_dt, end=end_dt)
pdf = df_hist.to_pandas()

print(f"✅ Success! Total candles: {len(df_hist)}")

# Professional Candlestick Chart
fig = go.Figure(data=[go.Candlestick(
    x=pdf['timestamp'],
    open=pdf['open'],
    high=pdf['high'],
    low=pdf['low'],
    close=pdf['close'],
    name='OHLC'
)])

fig.update_layout(
    title=f'{symbol} 24h Candlestick Chart (1m)',
    xaxis_title='Time',
    yaxis_title='Price (USD)',
    template='plotly_dark',
    xaxis_rangeslider_visible=True
)
fig.show()

Fetching 24h of 1m data for BNB-USD to visualize candlesticks...
✅ Success! Total candles: 1119


## 📊 Alpha Projection
Check basic correlation with forward returns.

In [4]:
df = session.run_alpha_score(df)
fig_scat = px.scatter(df.to_pandas(), x="sma_21", y="fwd_ret_1", trendline="ols", title="Alpha Signal Correlation")
fig_scat.update_layout(**PLOTLY_DARK)
fig_scat.show()

## 🛠 Ready for deep-dive?
To continue your research, navigate to the specialized folders or run `make bot-start` for live trading.